In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [ ]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
v1.dot(dv)

In [ ]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [ ]:
v2.dot(dv)

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
documents[10]

In [ ]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [ ]:
len(texts)

In [ ]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

In [ ]:
import numpy as np
X = np.array(vectors)

X.shape


In [ ]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

scores = X.dot(v_query)
# scores can be computed by below approach also
# scores = [v_query.dot(X[i]) for i in range(len(X))]

idx = np.argmax(scores)
idx, scores[idx]

In [ ]:
documents[idx]

In [ ]:
# top 5 results

top5 = np.argsort(scores)[-5:]

for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5,filter_dict={"course": "llm-zoomcamp"})

results

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
# RAG with text search
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

In [ ]:
# override RAGBASE with vectosearch
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
# Passing vindex instead of text based index
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [ ]:
vector_assistant.rag("the program has already begun, can I still sign up?")

In [ ]:
#initialize the index
from sqlitesearch import VectorSearchIndex

# sqlitesearch supports three ANN modes:
# lsh (default): up to 100K vectors, random hyperplane projections
# ivf: 10K-500K vectors, K-means clustering
# hnsw: 10K-1M+ vectors, proximity graph (highest recall)
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [ ]:
#indexing data using sqliteindex
vs_index.fit(vectors, documents)

In [ ]:
# perform the search

query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [ ]:
results

In [ ]:
# filter results based on course

results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [ ]:
results

In [ ]:
vs_index.close()